<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: GeoPandas Recap

## Learning Objectives

By the end of this session, you will be able to:

1. Explain how Census geography (FIPS codes, GEOIDs) organizes data, and use the Census API to pull both decennial and ACS estimates
2. Load, filter, and visualize geospatial data using GeoPandas
3. Apply appropriate coordinate reference systems and projections to geospatial datasets
4. Perform geocoding operations to convert addresses into geographic coordinates
5. Execute spatial joins using various spatial relationship operations
6. Create multi-layer maps to visualize spatial relationships between datasets

---

Adapted from CYPLAN101 Fall 2025 Lab 6.

**Data for this notebook** lives in `data/raw/` inside this week's folder: it's already there, no download needed. `data/raw/` is a read-only reference copy: please don't edit or overwrite files there. If you need to modify the data, load it into a variable (as below) and work with that copy instead.


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("geopandas_recap.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part I. Geopandas Basics and Loading Data
In this lab exercise, you will be producing a map containing three layers:  
  * The City of Berkeley Jurisdictional Boundary,
  * A choropleth map depicting Median Household Income for each census tract, and
  * Point locations of your favorite locations around campus.

To kick things off, you will install Geopandas and its dependencies, then load geospatial data from various file formats (Geopackage, GeoJSON). Then you will explore Geopandas capabilities and optionally learn to create interactive visualizations using Plotly Express.


### Step 1: Install Geopandas and Dependencies
Begin by installing Geopandas in your Python environment using the following command.  You can do this in the terminal or in a code cell.

```
pip install geopandas
```

  > **Hint:** What packages do you need to install in addition to installing geopandas?


In [ ]:
# Add GeoPandas and other dependencies and keys here
import os
import pandas as pd
from census import Census
from us import states
import matplotlib.pyplot as plt

# Census API key, loaded from an environment variable (same pattern used elsewhere this course)
try:
    census_api_key = Census(os.environ.get('CENSUS_API_KEY'))
except ValueError:
    census_api_key = None


### Step 2: Import Required Libraries
Import Geopandas and other necessary libraries using the code cell below.  

> **Hint:** You may need to return to this code cell over the span of this exercise to add dependencies.

In [ ]:
import geopandas as gpd

### Step 3: Download Berkeley Jurisdictional Boundary Data

As we have discussed, GIS data can be found in a variety of different formats. **For Example:**
  * GeoJSONs are a JSON-based open standard format for encoding geographic data structures. GeoJSON works well with web applications and all content (e.g., geometry + attributes) is stored in a single .geojson or .json file.

  * GeoPackages are an open-standard format based on SQLite. GeoPackages are single .gpkg files that can store multiple layers of vector and raster data, plus metadata. This is the format recommended by the [Open Geospatial Consortium](https://www.ogc.org/) (OGC).

  * **Shapefiles** are a format developed by ESRI in the 1990s that stores vector geographic data. A "shapefile" is actually a collection of multiple files that must stay together:
    - `.shp` - the geometry (points, lines, polygons)
    - `.shx` - the index file
    - `.dbf` - the attribute data (like a spreadsheet)
    - `.prj` - the coordinate reference system
    - Optional: `.cpg`, `.sbn`, `.sbx`, etc.

**Today we will be working with the GeoJSON file format when sourcing input data from the [City of Berkeley's Open Data Portal](https://data.cityofberkeley.info/).**  
Using the link provided above, search for, and download the latest version of the Berkeley jurisdictional boundary as a GeoJSON file.

> **Hint:** If the download fails, you can also find a copy of this file in the course files on the lab page.

This file is already provided at `data/raw/Land_Boundary_20251008.geojson` for this course. No need to download or upload it yourself.

### Step 4: Load the Geospatial Data
Read the Berkeley boundary file into a GeoDataFrame.  You can use the template code below to do this.

```
gdf = gpd.read_file('your_file_name.geojson')
```
Once you have loaded your data, preview it using `.plot()`

```
gdf.plot()
```

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️
In the cell below, adapt the template code to load your geojson file.  

**Be sure to remember this variable name!  You will need it again later in this lab exercise**

In [ ]:
# Load in your data here.
berkeley_bound = ...

# Plot your data here.
berkeley_bound.plot()

In [ ]:
grader.check("q_berkeley_load")

### Census Geography: FIPS Codes and GEOIDs

Before pulling any Census data, it helps to understand how the Census Bureau organizes geography, since every dataset you'll work with in this lab — decennial Census, ACS, TIGER shapefiles — uses the same identifiers to describe *where* a piece of data belongs.

**Geographic hierarchy:** Census geography nests inside itself — states contain counties, counties contain census tracts (~4,000 people each), tracts contain block groups (~1,500 people each). ACS does not publish data below the block-group level.

**FIPS codes** (Federal Information Processing Series) are standardized numeric codes the Census Bureau has used for over 30 years to identify states, counties, and other areas. They're assigned alphabetically within each level, and they nest: California's state FIPS is `06`, and Alameda County's county FIPS is `001` — so Alameda County's full FIPS code is `06001`.

**GEOIDs** (Geographic Identifiers) combine these nested FIPS codes into one string that uniquely identifies a place, and are what let you join demographic data (like ACS estimates) to spatial data (like TIGER tract boundaries):

| Area Type          | GEOID Structure                | Example (Alameda County, CA) |
| ------------------- | ------------------------------- | ------------------------------ |
| State               | STATE (2 digits)               | `06`                            |
| County              | STATE + COUNTY (2+3)           | `06001`                         |
| Census Tract        | STATE + COUNTY + TRACT (2+3+6) | `06001400100`                   |
| Census Block Group  | STATE + COUNTY + TRACT + BG    | `060014001001`                  |

You'll build a GEOID column yourself later in this lab, once you've pulled some real tract-level data.

### Your First Census Pull: Decennial Population Data

Let's put the Census API to use with the simplest possible pull: total population by tract, from the decennial census. This uses the `census_api_key` object you already set up in Step 1, plus two of its methods:

* `sf1` — pulls from the decennial census's Summary File 1
* `state_county_tract` — filters to a specific state, county, and tract

> **Curious how to query the Census API directly?** See the [package documentation](https://pypi.org/project/census/#description) or the Census Bureau's [Developers](https://www.census.gov/data/developers.html) page.

In [ ]:
pop_data = census_api_key.sf1.state_county_tract(
    fields=['P001001'],
    state_fips='06',   # California
    county_fips='001',  # Alameda County
    tract='*',
    year=2010
)

df_pop = pd.DataFrame(pop_data)
print(df_pop.head())

This is the same API object, the same `state_county_tract` pattern, and the same Census geography you just learned about — decennial data is simpler because it doesn't need a `fields=` code lookup beyond total population. ACS, which you'll use next, works the same way but returns many more estimated variables.

---

### Step 5: Load Your Census Data
In this step, you will again use the Census API to load in your data. This time, however, you will be combining tract tabular data (attributes) with tract geometry (features).

  > **Note:** If the Census API is not working, you have two options.  While you can download the data as a GeoJSON file and tabular data from the course files, I would strongly encourage you to try extracting the data from IPUMS NHGIS database. **We will cover this at a later time.** This will require you to apply a slightly different workflow.

**To complete this step, you will need to install/import several dependencies first:**

```
import os
import pandas as pd
from census import Census
from us import states
```
Using the Step 1 code cell above (and/or the terminal) please load in these dependencies.

**You will also need your Census API key available as the `CENSUS_API_KEY` environment variable.  As a reminder, the code for this will look like this:**

```
try:
    census_api_key = Census(os.environ.get('CENSUS_API_KEY'))
except ValueError:
    census_api_key = None
```

**You can do this in the Step 1 code cell as well.  If the Census API isn't reachable, `data/raw/alameda_tracts23.csv` is a pre-fetched fallback with the same shape as the API response.**

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️
In the code cell below, adapt the code  to retrieve the **2023 ACS 5-year Estimates for Population (B01003_001E) and Median Household Income (B19013_001E) for all census tracts in Alameda County**

<!-- BEGIN QUESTION -->



In [ ]:
# Your code here: pull 2023 ACS 5-year estimates for population and median household
# income for all census tracts in Alameda County (fields B01003_001E, B19013_001E)

<!-- END QUESTION -->



In [ ]:
print(alameda_dataframe.head())

### A quick check before you move on

Recall from last week's lab: ACS estimates sometimes come back as `-666666666` -- the Census Bureau's code for "this tract's estimate is suppressed or too unreliable to publish," not a real income value. It doesn't affect the map you're building today (that one's colored by `population`, not income), but `median_household_income` is sitting right there in your dataframe, and you'll likely reach for it in your own final project. Check it and filter it out now, while you remember why it's there.

In [ ]:
print(alameda_dataframe['median_household_income'].describe())

# Drop rows where the estimate is suppressed/unreliable (the -666666666 sentinel)
alameda_dataframe = alameda_dataframe[alameda_dataframe['median_household_income'] > 0]

Now that you have loaded in your tabular data, its time to retrieve the geometry and join it to the tabular data. To do this, you will need to construct a GEOID field and then use geopandas to download census tract geometry.

Recall the GEOID structure from earlier in this lab (`state` + `county` + `tract`). Based on the columns from your extracted census tract data, you can perform a vectorized operation to create a new GEOID column.

In [ ]:
# Use this space to create a new GEOID column
alameda_dataframe['GEOID'] = alameda_dataframe['state'] + alameda_dataframe['county'] + alameda_dataframe['tract']

print(alameda_dataframe['GEOID'].head())


Now that you have a field upon which to base your join/merge, you can go ahead and retrieve the census tract geometries.

To do this, navigate to the [census TIGER file FTP site](https://www2.census.gov/geo/tiger/) and locate the zip file for tracts in California that corresponds to 2023 data.

> **Hint:** Grab the URL for the ZIP file located at...TIGER2023 > TRACT > ...WHAT FIPS CODE IS THE STATE OF CALIFORNIA?

In [ ]:
# Read in the Census TIGER file containing the geometry for each tract in the State of California
ca_tracts = gpd.read_file('data/raw/tl_2023_06_tract.zip')


# Join census data with geometries
alameda_census_gdf = ca_tracts.merge(alameda_dataframe, on='GEOID', how='right')
print(alameda_census_gdf.columns)

# Plot the result
alameda_census_gdf.plot()


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️
In the cell below, perform the following tasks to explore the data.  **Be sure to use print statements to ensure the results are displayed in the output.**

  * Print the first 5 rows of the GeoDataFrame
  * Identify the types of each column in the GeoDataFrame
  * View basic **information** about the GeoDataFrame
  * Check the **geometry** of the GeoDataFrame
  * Check the crs of the GeoDataFrame

  > **Hint:** You can use the `geometry` method and the `crs` method to retrieve the last two items on this list.



<!-- BEGIN QUESTION -->



In [ ]:
# Use this space to print the first 5 rows of the GeoDataFrame
...

# Use this space to identify the types of each column in the GeoDataFrame
...

# Use this space to view basic information about the GeoDataFrame
...

# Use this space to check the geometry of the GeoDataFrame
...

# Use this space to check the crs of the GeoDataFrame
...

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part II. Geocoding and Projections

In this section you will learn how vector geospatial data (points, lines, polygons) is stored.  You will also check and set coordinate systems and perform spatial filtering to select census tracts within Berkeley and geocode addresses from your **Assignment #1** ("My Places") CSV file.


### Step 1: Geocode your own data
Thus far, you have prepared data for two of the three layers to be included in our map, namely the Berkeley jurisdictional boundary and the Berkeley tracts.  Now you will bring in your own original data using a process called **geocoding**.

Geocoding is the process of converting text-based location descriptions (like addresses or place names) into geographic coordinates (latitude and longitude) that can be mapped and analyzed spatially.



<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️
Load your Assignment #1 CSV file (the `{firstName}_{sid}_A1.csv` file you built and saved in Assignment 1) and edit the code cell below to geocode it.

For this step, you can use the following template code:

```
geo_coded_data = gpd.tools.geocode(addresses)

geo_coded_data
```


<!-- BEGIN QUESTION -->



In [ ]:
# Use this space to read in your Assignment 1 submission CSV
# Replace with your own file: {firstName}_{sid}_A1.csv
# a1_submission = pd.read_csv(...)
# print(a1_submission.head())

# Use this space to apply the gpd.tools.geocode() function to your data
# Hint: Use the argument timeout = 10 to circumvent timeout error issues
# geo_coded_data = ...

<!-- END QUESTION -->

### Step 2: Vector Data Types and Projections

In this exercise we are working with several different **geospatial datatypes** and two different **Coordinate Reference Systems (CRS)**.

#### Geospatial Data Types
We are contending with both point and polygon data:

Recall the three types of vector geometries:

* **Points** - stored as single coordinate pairs (x, y)
* **Lines** - stored as ordered sequences of coordinate pairs
* **Polygons** - stored as closed sequences of coordinate pairs (first point = last point)


#### Coordinate Reference Systems (CRS)
 Recall that a CRS is a framework that defines how geographic coordinates (like latitude and longitude) relate to actual locations on Earth's surface, specifying the projection, datum, and units used to represent spatial data.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️
Use the code cell below to check (again) the coordinate reference systems for each of your GeoDataFrames.  If you are unsure which variable corresponds to a GeoDataFrame, you can check this easily using the **Variables** button in the bottom lefthand corner of the browser viewport.

Do you notice a discrepancy in the CRS within any of the GeoDataFrames?  In addition to the foregoing task, you should also correct this discrepancy in the code cell below. You can use the following template code to do this:

```
gdf = gdf['geometry'].to_crs('COORDINATE REFERENCE SYSTEM HERE')
```

In [ ]:
# Use this space to print the CRS of the Berkeley jurisdictional boundary
print(berkeley_bound.crs)

# Use this space to print the CRS of tracts within Alameda County
print(alameda_census_gdf.crs)

# Use this space to print the CRS of your geocoded point locations from Assignment #1
print(geo_coded_data.crs)

# Use this space to update any CRS that does not match the other CRS from our prospective map layers.
alameda_census_gdf = ...

print(alameda_census_gdf.crs)

In [ ]:
grader.check("q_crs")

### Step 3: Simple Spatial Filter - Select Census Tracts Within Berkeley

Recall that during the previous lab exercise you performed a tabular join using the merge function.

Today, you will perform a spatial join based on geographic location of two GeoDataFrames. A spatial join can occur between layers of many types. For example, you might perform a spatial join to identify points representing locations where a taxi driver picked up an individual within a particular zip code.

Recall that to perform a spatial join you would use `gdf.sjoin()`

Arguments for this function include:
  * The reference GDF
  * The join type (e.g., `how='inner'`)
  * The predicate (e.g., `predicate = 'within'` - or possibly...'contains', 'touches', 'crosses', 'overlaps')

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️

Using the template code and guidance above, write a spatial join that creates a new GeoDataFrame containing only the  tracts that intersect with Berkeley's jurisdictional bounds.  


In [ ]:
# Use this space to create your join and new GeoDataFrame
berkeley_census_gdf = ...

# Use this space to plot the result
berkeley_census_gdf.plot()

In [ ]:
grader.check("q_join")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part III. Basic Visualization and Spatial Analysis
In this final part of the lab, you will create static maps using matplotlib and overlay your three GeoDataFrames.

### Step 1: Creating a Map Using matplotlib

You may have noticed that the .plot function provides only basic functionality.  There are many plotting libraries available for maps, but today you will continue using the default, **matplotlib**, a common and versatile Python plotting library.

First we will be updating the **symbology** for just one of our three layers to improve visualization.  

You may use the following code template to do this.

```
gdf.plot(facecolor=, edgecolor=, linewidth=)
plt.show()
```
  > **Note:** `plt.show()` displays the matplotlib figure/plot in a window or inline in a Jupyter notebook, rendering all the plotting commands that were called before it.
  
  <br>

  > **Hint:** You can use the [matplotlib cheat sheet](https://matplotlib.org/cheatsheets/_images/cheatsheets-1.png) and [documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html) to determine the appropriate arguments for this.  

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️
Using the template code above, plot only your Berkeley jurisdictional boundary GDF.

**You should apply a black outline and a white or transparent fill.**

<!-- BEGIN QUESTION -->



In [ ]:
# Use this code cell to adapt the template code and plot only the Berkeley jurisdictional
# boundary with a black outline and white or transparent fill
...

<!-- END QUESTION -->

### Step 2: Creating Maps with Multiple Layers Using matplotlib and downloading plots
One of the benefits of GIS and mapping in general is the ability to layer multiple types of data upon one another to add context and convey information. Matplotlib has this functionality In this final step you will be applying the following template code to do just that with all three of your layers.  

```
fig, ax = plt.subplots()

gdf1.plot(ax=ax)
gdf2.plot(ax=ax)
gdf3.plot(ax=ax)

plt.show()
```

To download the result, simply add
`plt.savefig('filename.png')` before `plt.show()`.

  > **Hint:** Be careful about the layer draw order!

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ❗️ Your Turn ❗️
Using the template code above, plot all three of your layers using the following symbology:

1. Berkeley jurisdictional boundary - apply a black outline and white fill
2. Berkeley Census Tract Boundaries - as a choropleth layer (Hint: `gdf.plot(column=, cmap=, legend=True)`)
3. Plot the point locations from your Assignment #1 submission - symbolize according to the location type (e.g., restaurant, study spot etc.)


**Download the resulting map as a PNG file.**


<!-- BEGIN QUESTION -->



In [ ]:
...

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary: Building a Multi-Layer Map with GeoPandas

| Step | Takeaway |
|---|---|
| `gpd.read_file()` on a GeoJSON | Loads a jurisdictional boundary into a `GeoDataFrame` ready to plot |
| Census API + ACS sentinel `-666666666` | Suppressed estimates must be filtered out before they distort a map or plot |
| `GEOID` from `state` + `county` + `tract` | The join key that links tabular Census data to TIGER/Line tract geometry |
| `gpd.tools.geocode()` | Converts text addresses into point geometry for your own data |
| Checking `.crs` on every layer | Layers must share a CRS before they can be joined or plotted together correctly |
| `gdf.sjoin(other, predicate='intersects')` | Selects features in one layer based on their spatial relationship to another |
| `fig, ax = plt.subplots()` + repeated `gdf.plot(ax=ax, ...)` | Combines multiple GeoDataFrames into one multi-layer static map |

## Key Takeaways

1. **Always check and reconcile CRS across layers** before joining or plotting them together - a mismatched CRS silently misaligns your map.
2. **A spatial join (`sjoin`) filters or matches features by geometry**, the same way a tabular merge matches on a shared key.
3. **Geocoding turns addresses into point geometry** that can be layered onto the same map as polygon and choropleth data.
4. **Layer order matters in matplotlib** - draw polygons first, then boundaries, then points, so nothing is hidden underneath another layer.
